# 08 — Metadata Generation

**Marker:** `NOTEBOOK_08_METADATA_GENERATION_FRESH_V1`

This notebook converts the newest checked script into publication-ready
metadata while preserving the fact checker's corrections.

It generates:

- one primary title
- three to five alternate titles
- a short post caption
- a concise description
- thumbnail text
- hashtags
- search keywords
- category and audience level
- deterministic fact-check and source traceability

Public-facing text does not mention the fact-checking process. Review status
and source URLs remain available in the saved JSON for the pipeline.


## Load the project

In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "educational_shorts").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the project root containing educational_shorts/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.metadata import (
    build_metadata_filename,
    find_checked_script_file,
    find_fact_check_report_file,
    generate_metadata,
    load_fact_check_report,
    load_script,
    save_metadata,
    summarize_metadata,
)
from educational_shorts.prompts import load_prompt

print("NOTEBOOK_08_METADATA_GENERATION_FRESH_V1")
print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
CHECKED_SCRIPTS_DIRECTORY = PROJECT_ROOT / "data" / "checked_scripts"
FACT_CHECKS_DIRECTORY = PROJECT_ROOT / "data" / "fact_checks"
METADATA_DIRECTORY = PROJECT_ROOT / "data" / "metadata"

# Leave as None to use the newest checked script and its matching report.
CHECKED_SCRIPT_FILENAME = None
FACT_CHECK_REPORT_FILENAME = None

TEMPERATURE = 0.5
GENERATION_SEED = 42
MAX_ATTEMPTS = 4

print(f"Checked scripts: {CHECKED_SCRIPTS_DIRECTORY}")
print(f"Fact-check reports: {FACT_CHECKS_DIRECTORY}")
print(f"Metadata output: {METADATA_DIRECTORY}")

## Load the checked script and matching fact-check report

In [ ]:
checked_script_path = find_checked_script_file(
    scripts_directory=CHECKED_SCRIPTS_DIRECTORY,
    filename=CHECKED_SCRIPT_FILENAME,
)

checked_script = load_script(checked_script_path)

fact_check_report_path = find_fact_check_report_file(
    reports_directory=FACT_CHECKS_DIRECTORY,
    script=checked_script,
    filename=FACT_CHECK_REPORT_FILENAME,
)

fact_check_report = load_fact_check_report(fact_check_report_path)

print(f"Checked script: {checked_script_path}")
print(f"Fact-check report: {fact_check_report_path}")
print(f"Topic: {checked_script.topic.title}")
print(f"Words: {checked_script.word_count}")
print(f"Seconds: {checked_script.estimated_total_seconds}")
print(f"Fact-check verdict: {fact_check_report.verdict}")
print(
    f"Manual review recommended: "
    f"{fact_check_report.requires_manual_review}"
)

## Load the metadata prompt

In [ ]:
metadata_system_prompt = load_prompt("metadata_generation")
print("Metadata prompt loaded.")

## Generate and validate metadata

In [ ]:
metadata = generate_metadata(
    script=checked_script,
    report=fact_check_report,
    system_prompt=metadata_system_prompt,
    source_script_filename=checked_script_path.name,
    fact_check_report_filename=fact_check_report_path.name,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
    max_attempts=MAX_ATTEMPTS,
)

for name, value in summarize_metadata(metadata).items():
    print(f"{name}: {value}")

## Preview public-facing metadata

In [ ]:
print(f"TITLE\n{metadata.title}\n")

print("ALTERNATE TITLES")
for index, title in enumerate(metadata.alternate_titles, start=1):
    print(f"{index}. {title}")

print(f"\nTHUMBNAIL TEXT\n{metadata.thumbnail_text}")

print(f"\nSHORT CAPTION\n{metadata.short_caption}")

print(f"\nDESCRIPTION\n{metadata.description}")

print("\nHASHTAGS")
print(" ".join(metadata.hashtags))

print("\nKEYWORDS")
print(", ".join(metadata.keywords))

print(f"\nCATEGORY\n{metadata.category}")
print(f"\nAUDIENCE LEVEL\n{metadata.audience_level}")

## Preview internal review and source fields

In [ ]:
print(f"Fact-check verdict: {metadata.fact_check_verdict}")
print(f"Requires manual review: {metadata.requires_manual_review}")
print(f"Fact-check summary: {metadata.fact_check_summary}")

print("\nSource URLs used by verification:")
if metadata.source_urls:
    for url in metadata.source_urls:
        print(f"- {url}")
else:
    print("No verification URLs were recorded.")

## Save metadata

In [ ]:
metadata_path = (
    METADATA_DIRECTORY
    / build_metadata_filename(metadata)
)

save_metadata(
    metadata=metadata,
    output_path=metadata_path,
)

print(f"Saved metadata to: {metadata_path}")

## Preview complete saved object

In [ ]:
print(metadata.model_dump_json(indent=2))